# Decode the Bioreactor — 03 Hybrid: base model + ML correction
**Idea**: keep a base prediction and let a small ML model learn what the base gets wrong:

  y_hat(t) = base(t) · exp(g(features))

- **base** = mechanistic model (from notebook 02) **or** the median training curve
- **g** learns the *relative* error of the base (log ratio, consistent with MAPE) from: time, day-0 values, cumulative feed volumes up to t, volume at t, base prediction. All are known for a test batch.
- **methods**: `ridge_day` (one ridge regression per state and day; linear, interpretable) and `gbr` (shallow gradient-boosted trees; non-linear).
- Corrections are clipped to ±40 %.

Four variants are compared on the same CV folds: `median+ridge_day`, `median+gbr`, `mech+ridge_day`, `mech+gbr` (plus the plain bases).
Code: `src/bioreactor/hybrid.py`, metrics: `src/bioreactor/evaluate.py`.

**Leakage note**: in each fold the mechanistic parameters and the ML correction are fitted on the training batches only. The ML is trained on the mechanistic model's *in-sample* residuals, which are a bit smaller than on new batches — CV tells the honest result.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from bioreactor.data import load, to_arrays, STATES
from bioreactor.model import predict_at_obs, NAMES
from bioreactor.fit import fit_staged, initial_from_data
from bioreactor.hybrid import median_curve_preds, feature_table, feature_columns, ResidualCorrector, to_list
from bioreactor.evaluate import metrics, mape_row

df, tmpl, train_ids, test_ids = load(ROOT / "data")
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")
dev_ids = splits.query("role == 'dev'").batch_id.tolist()
hold_ids = splits.query("role == 'holdout'").batch_id.tolist()
fold_of = splits.query("role == 'dev'").set_index("batch_id").fold.to_dict()

# mechanistic fit settings (same meaning as in notebook 02)
MAX_NFEV_STAGE, MAX_NFEV, N_STARTS = 20, 60, 1
FIT_KW = dict(residual="log", loss="soft_l1", f_scale=0.1)
METHODS = ["ridge_day", "gbr"]
CACHE = ROOT / "outputs" / "params_cache"; CACHE.mkdir(parents=True, exist_ok=True)
USE_CACHE = True   # reuse mechanistic parameters already fitted for the same training set

In [ ]:
def mech_params(ids, tag):
    """Fit the mechanistic model on `ids` (staged fit), or load it from the cache."""
    f = CACHE / f"{tag}.csv"
    if USE_CACHE and f.exists():
        s = pd.read_csv(f, index_col=0).iloc[:, 0]
        if list(s.index) == NAMES:
            print(f"[{tag}] mechanistic parameters loaded from cache"); return s.values
    t0 = time.time()
    p, _ = fit_staged(to_arrays(df, ids), None, p_init=initial_from_data(df, ids),
                      max_nfev_stage=MAX_NFEV_STAGE, max_nfev_final=MAX_NFEV,
                      n_starts=N_STARTS, verbose=False, **FIT_KW)
    pd.Series(p, index=NAMES).to_csv(f); print(f"[{tag}] mechanistic fit {time.time()-t0:.0f} s")
    return p

def evaluate_split(tr_ids, va_ids, tag, methods=METHODS):
    """Fit everything on tr_ids, evaluate all variants on va_ids. Returns (MAPE table, objects)."""
    a_tr, a_va = to_arrays(df, tr_ids), to_arrays(df, va_ids)
    p = mech_params(tr_ids, tag)
    bases = {"median": (median_curve_preds(df, tr_ids, a_tr), median_curve_preds(df, tr_ids, a_va)),
             "mech":   (predict_at_obs(p, a_tr), predict_at_obs(p, a_va))}
    rows, objs = {}, {"p": p, "a_va": a_va}
    for bname, (b_tr, b_va) in bases.items():
        rows[bname] = mape_row(a_va, b_va)
        objs[bname] = (None, b_va)
        F_tr, F_va = feature_table(df, a_tr, b_tr), feature_table(df, a_va, b_va)
        for m in methods:
            corr = ResidualCorrector(m).fit(F_tr)
            pred = to_list(F_va, corr.predict(F_va), a_va)
            rows[f"{bname}+{m}"] = mape_row(a_va, pred)
            objs[f"{bname}+{m}"] = (corr, pred)
    return pd.DataFrame(rows).T, objs

## 1. One fold (quick look)
MAPE % on the validation fold for every variant. The first run fits the mechanistic model for this fold (a few minutes); later runs load it from `outputs/params_cache/`.

In [ ]:
FOLD = 0
tr_ids = [b for b in dev_ids if fold_of[b] != FOLD]
va_ids = [b for b in dev_ids if fold_of[b] == FOLD]
t0 = time.time()
tab, objs = evaluate_split(tr_ids, va_ids, tag=f"fold{FOLD}")
print(f"{time.time()-t0:.0f} s")
tab.round(2).sort_values("all")

Where do the variants miss? Mean absolute % error by day on the validation fold.

In [ ]:
def abs_err_by_day(arr, preds):
    Y = np.vstack(arr.obs_y); Yh = np.vstack(preds); T = np.round(np.concatenate(arr.obs_t)).astype(int)
    e = pd.DataFrame(100 * np.abs(Yh - Y) / Y, columns=STATES); e["day"] = T
    return e.groupby("day").mean()

SHOW = ["median", "mech", "median+ridge_day", "mech+ridge_day", "median+gbr", "mech+gbr"]
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for name in SHOW:
    e = abs_err_by_day(objs["a_va"], objs[name][1])
    for a, s in zip(ax, STATES):
        a.plot(e.index, e[s], "o-" if "+" in name else "s--", ms=3, label=name)
for a, s in zip(ax, STATES):
    a.set_title(s); a.set_xlabel("day"); a.set_ylabel("|% error|")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

### Interpretation
**Ridge (per day)**: standardised coefficients — how much each input shifts the prediction (log units ≈ relative change) per 1 std of that input, by day. Shown for the median base: the coefficients then describe directly how inputs move a batch away from the typical curve.

In [ ]:
corr_r = objs["median+ridge_day"][0]
for s in ["vcd", "byproduct"]:
    C = corr_r.ridge_coefficients(s)
    C = C[[c for c in C.columns if not c.startswith("logbase_")]]   # base is constant per day for the median base
    plt.figure(figsize=(10, 4)); lim = np.abs(C.values).max()
    plt.imshow(C.values.T, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
    plt.yticks(range(C.shape[1]), C.columns); plt.xticks(range(C.shape[0]), C.index); plt.xlabel("day")
    plt.colorbar(label="std. coefficient (log units)"); plt.title(f"ridge correction, median base: {s}")
    plt.tight_layout(); plt.show()

In [ ]:
# gradient boosting: feature importances per state (mechanistic base)
corr_g = objs["mech+gbr"][0]
imp = pd.DataFrame({s: corr_g.models[s].feature_importances_ for s in STATES}, index=feature_columns("vcd", "gbr"))
imp.sort_values("vcd", ascending=False).round(3)

## 2. 5-fold cross-validation (all variants)
The honest comparison. Each fold: mechanistic fit + ML fit on its training batches only. Mechanistic parameters are cached per fold, so re-runs (e.g. after changing the ML part) are fast.

In [ ]:
RUN_CV = False
if RUN_CV:
    tabs = []
    for f in range(5):
        tr_ = [b for b in dev_ids if fold_of[b] != f]; va_ = [b for b in dev_ids if fold_of[b] == f]
        t_, _ = evaluate_split(tr_, va_, tag=f"fold{f}")
        tabs.append(t_.assign(fold=f)); print(f"fold {f}: pooled MAPE", t_["all"].round(2).to_dict())
    cv = pd.concat(tabs).rename_axis("variant").reset_index()
    summary = cv.groupby("variant")[STATES + ["all"]].agg(["mean", "std"]).round(2)
    display(summary.sort_values(("all", "mean")))

## 3. Hold-out check — run ONCE, after choosing the variant from CV
Fit on all 39 dev batches, evaluate on the 10 hold-out batches.

In [ ]:
RUN_HOLDOUT = False
if RUN_HOLDOUT:
    tab_h, objs_h = evaluate_split(dev_ids, hold_ids, tag="dev_all")
    display(tab_h.round(2).sort_values("all"))

## 4. Final: fit on all 49 batches → submission
Pick the variant (best CV), then predict the 12 test batches → `outputs/submission_hybrid_<base>_<method>.csv`.

In [ ]:
RUN_FINAL = False
BASE, METHOD = "mech", "ridge_day"     # set from the CV result
if RUN_FINAL:
    a_all = to_arrays(df, train_ids)
    a_test = to_arrays(df, test_ids, sample_mask="predict")
    if BASE == "mech":
        p_all = mech_params(train_ids, tag="all49")
        b_all, b_test = predict_at_obs(p_all, a_all), predict_at_obs(p_all, a_test)
    else:
        b_all, b_test = median_curve_preds(df, train_ids, a_all), median_curve_preds(df, train_ids, a_test)
    corr = ResidualCorrector(METHOD).fit(feature_table(df, a_all, b_all))
    F_test = feature_table(df, a_test, b_test)
    preds = to_list(F_test, corr.predict(F_test), a_test)
    failed = [b for b, y in zip(a_test.ids, preds) if np.isnan(y).any()]
    assert not failed, f"NaN predictions for {failed}"

    pred_of = {b: (ts, ys) for b, ts, ys in zip(a_test.ids, a_test.obs_t, preds)}
    sub = tmpl.copy(); sub[STATES] = sub[STATES].astype(float)   # template columns are int zeros
    for b, g in sub.groupby("batch_id", sort=False):
        ts, ys = pred_of[b]; order = np.argsort(ts)
        assert len(g) == len(ts) and np.allclose(np.sort(g.elapsed_time.values), ts[order], atol=1e-3), b
        sub.loc[g.sort_values("elapsed_time").index, STATES] = ys[order]
    assert sub[STATES].notna().all().all()
    out = ROOT / "outputs" / f"submission_hybrid_{BASE}_{METHOD}.csv"
    sub.to_csv(out, index=False); print("written", out)
    display(sub.head())